# 🟨 Caderno 03: Engenharia de Features & Fábrica de Recursos (Camada Gold)

## 🎯 1. Objetivo do Notebook

Este caderno funciona como a **Fábrica de Recursos (Feature Engineering)** do nosso ecossistema de MLOps. O objetivo central é consumir a base de dados consolidada, auditada e limpa da camada Silver (`df_silver_2025.parquet` extraída do AWS S3) e transformá-la estruturalmente. 

Aqui, os dados brutos são lapidados para gerar **três conjuntos de dados específicos e independentes**. Modelos preditivos possuem naturezas matemáticas distintas; portanto, este notebook é o coração arquitetural que garante o input perfeito para cada algoritmo, blindando o pipeline contra vieses e vazamento de dados (*Data Leakage*).

```text
                               ┌──> df_serie_temporal.csv ───> (04_modelo_forecast)
                               │
[Data Lake S3: Silver 2025] ───┼──> df_matriz_risco.csv  ───> (05_modelo_ola_risk)
                               │
                               └──> df_perfis_cluster.csv ───> (06_clusterizacao)

## 📖 Dicionário de Dados Completo - Entrada Camada Gold (`df_gold`)

Este dicionário detalha a estrutura exata das **26 colunas** contidas no arquivo `df_silver_2025.parquet` carregado do AWS S3. Ele serve como o mapa oficial de engenharia para orientar a seleção de variáveis nos próximos blocos de código.

---

| Índice | Nome da Coluna | Tipo de Dado | Origem do Campo | Tratamento de Nulos Aplicado | Descrição e Regra de Negócio |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **0** | `Número` | `string` | Original (ITSM) | Não possui | Identificador único do incidente (Ex: INC0028941). Deve ser descartado nos modelos por não possuir peso estatístico. |
| **1** | `Prioridade` | `string` | Original (ITSM) | Não possui | Texto descritivo da criticidade do chamado (Ex: "1 - Crítica", "3 - Moderada"). |
| **2** | `Produto` | `string` | Original (ITSM) | Não possui | Nome da aplicação ou ativo de TI impactado (Ex: "E-mail", "Servidor AWS"). |
| **3** | `Categoria` | `string` | Original (ITSM) | Não possui | Classificação do tipo de falha técnica (Ex: "Acesso", "Hardware", "Software"). |
| **4** | `Subcategoria` | `string` | Original (ITSM) | Imputado: `'Não Informada'` | Detalhamento da categoria. Lacunas de preenchimento viraram uma categoria própria para a IA mapear erros de triagem. |
| **5** | `Grupo_designado` | `string` | Original (ITSM) | Não possui | Equipe responsável pelo atendimento do chamado (Ex: "Team07", "Team08"). Feature de alto impacto para o risco de SLA. |
| **6** | `Item_de_configuração` | `string` | Original (ITSM) | Imputado: `'Não Cadastrado'` | Ativo de infraestrutura específico associado ao incidente. Nulos foram tratados para mapear chamados genéricos. |
| **7** | `Aberto` | `datetime64[ns]`| Original (ITSM) | Não possui | Carimbo de data e hora (Timestamp) exato do momento de abertura do ticket. |
| **8** | `Resolvido` | `datetime64[ns]`| Original (ITSM) | Mantido: `NaT` (Not a Time) | Timestamp da solução técnica. Mantido nulo para chamados que fecharam o ano ainda ativos (comportamento real). |
| **9** | `Encerrado` | `datetime64[ns]`| Original (ITSM) | Não possui | Timestamp do fechamento administrativo do ticket. |
| **10** | `Duração` | `Int64` | Original (ITSM) | Não possui | Tempo bruto total de vida útil do chamado expresso em segundos. |
| **11** | `Código_de_fechamento`| `string` | Original (ITSM) | Imputado: `'Não Encerrado'` | Categoria final do desfecho do chamado (Ex: "Resolvido pelo Usuário", "Sucesso"). |
| **12** | `Descrição_resumida` | `string` | Original (ITSM) | Não possui | Texto livre descritivo digitado pelo usuário ou gerado pelo robô de monitoramento. |
| **13** | `Solução` | `string` | Original (ITSM) | Imputado: `'Sem Descrição'` | Texto descritivo preenchido pelo técnico informando a tratativa aplicada para resolver o incidente. |
| **14** | `Aberto_por` | `string` | Original (ITSM) | Não possui | Identificação da conta que registrou o chamado (técnico, usuário ou ID do robô). |
| **15** | `Incidente_Pai` | `string` | Original (ITSM) | Imputado: `'Independente'` | ID do chamado correlacionado. Se preenchido, o chamado atual é um "filho dependente" de um problema central. |
| **16** | `Status` | `string` | Original (ITSM) | Não possui | Estado final do chamado no momento da extração (Ex: "Encerrado", "Resolvido"). |
| **17** | `Entrou_para_KPI?` | `string` | Original (ITSM) | Não possui | Flag textual indicando se o chamado era elegível contratualmente para metas de atendimento ("SIM" ou "NAO"). |
| **18** | `KPI_Violado?` | `string` | Original (ITSM) | Mantido nulo original | Flag textual indicando estouro de SLA ("SIM" ou "NAO"). Possui nulos legítimos para chamados isentos por regra. |
| **19** | `KPI_Status_Int` | `int32` | Engineered Feature | Não possui | Tradução numérica controlada de 'KPI_Violado?': 1 = SIM, 0 = NAO, -1 = Casos Omissos/Nulos. |
| **20** | `Prioridade_Num` | `float64` | Engineered Feature | Não possui | Isolamento numérico da criticidade do chamado (valores de 1.0 a 5.0), blindando contra quebras de texto. |
| **21** | `Possui_Pai` | `int32` | Engineered Feature | Não possui | Variável binária de hierarquia: 1 = O chamado possui um incidente pai associado; 0 = O chamado é independente. |
| **22** | `Duracao_Horas` | `Float64` | Engineered Feature | Não possui | Métrica de tempo de ciclo convertida de segundos para horas brutas de calendário (tempo corrido 24x7). |
| **23** | `Exige_Intervencao` | `boolean` | Engineered Feature | Não possui | Filtro mestre operacional: True = Exigiu esforço técnico humano; False = Ruído/Alarme de monitoramento automático. |
| **24** | `Data_Abertura` | `datetime64[ns]`| Engineered Feature | Não possui | Componente de data pura (Ano-Mês-Dia) extraído de 'Aberto'. Usado como âncora para os agrupamentos de tempo. |
| **25** | `Target_Risco_SLA` | `int32` | **Target Final (Y)** | Não possui | A variável alvo definitiva para o modelo de classificação. Consolida as regras comerciais e corrige os furos sistêmicos: 1 = Chamado Violou o SLA; 0 = Chamado Entregue no Prazo ou Isento. |

📐 3. Diretrizes de Engenharia por Modelo de Destino
📈 3.1. Dataset de Série Temporal Agrupado (Modelo 04 - Forecast)
Arquivo de Saída: df_serie_temporal.csv

O que criar: Um dataframe transformado via agrupamento cronológico contendo a contagem diária ou horária de incidentes.

Filtros Aplicados: Exige_Intervencao == True (Apenas esforço real humano).

Variáveis Inclusas: Data_Abertura (ou componentes de hora) e a contagem volumétrica (Total_Chamados).

Justificativa Técnica: Modelos de Forecasting (Prophet/ARIMA) exigem uma linha do tempo contínua e sequencial para identificar padrões matemáticos de sazonalidade (como o pico observado nas quintas-feiras e vales nos finais de semana).

Justificativa de Negócio: Subsidia o Planejamento de Capacidade (Capacity Planning). O gestor descobre o volume total de trabalho que entrará na fila física, permitindo otimizar a escala de técnicos.

🎯 3.2. Matriz de Atendimento Inicial (Modelo 05 - Classificação de Risco SLA)
Arquivo de Saída: df_matriz_risco.csv

O que criar: Uma tabela ao nível de incidente individual contendo estritamente as variáveis conhecidas no primeiro minuto de abertura do chamado.

Filtros Aplicados: Exige_Intervencao == True.

Variáveis Inclusas: Prioridade_Num, Grupo_designado, Categoria, Subcategoria, Possui_Pai e o alvo Target_Risco_SLA.

⚠️ Regra de Ouro (Data Leakage): É mandatório excluir as colunas Duracao_Horas, Status, Resolvido e Código_de_fechamento.

Justificativa Técnica: Se variáveis de encerramento forem mantidas no treino, o modelo sofrerá de vazamento de dados. Ele se tornará um adivinho do passado (se o chamado durou 20 horas, é óbvio que violou), perdendo o poder de generalização na produção.

Justificativa de Negócio: Permite uma atuação preventiva. O modelo calculará a probabilidade de estouro de SLA nos primeiros 5 minutos de vida do ticket, disparando alertas na tela do gestor antes que o contrato comercial seja violado e gere multas.

🤖 3.3. Matriz de Perfis Operacionais (Modelo 06 - Clusterizacao)
Arquivo de Saída: df_perfis_cluster.csv

O que criar: Uma matriz densa de comportamento combinando variáveis de abertura com métricas de desfecho final do incidente.

Filtros Aplicados: Exige_Intervencao == True.

Variáveis Inclusas: Diferente do Modelo 2, aqui você DEVE incluir a causa (Grupo_designado, Categoria), o efeito operacional (Duracao_Horas) e o efeito de compliance (Target_Risco_SLA), além da estrutura (Possui_Pai).

Transformações Exigidas: Aplicação de MinMaxScaler na coluna Duracao_Horas (para trazê-la à escala de 0 a 1) e One-Hot Encoding ou Target Encoding nas variáveis textuais.

In [ ]:
import pandas as pd
import numpy as np
import awswrangler as wr

print("📥 Conectando ao AWS S3 e extraindo a Camada Silver...")
print("-" * 60)

# Caminho oficial do seu Data Lake centralizado
s3_silver_path = "s3://aiops-locaweb-datalake-2026/silver/incidents_silver_2025.parquet"

# Leitura direta e otimizada do arquivo Parquet via AWS Wrangler
df_gold = wr.s3.read_parquet(path=s3_silver_path)

print(f"✅ Download concluído com sucesso!")
print(f"Total de registros humanos de Esforço Real carregados: {len(df_gold)}")
print(f"Estrutura da Matriz: {df_gold.shape[0]} linhas x {df_gold.shape[1]} colunas")
print("-" * 60)

# Amostragem de segurança para validar as novas features
df_gold[['Data_Abertura', 'Duracao_Horas', 'Possui_Pai', 'Target_Risco_SLA']].head()